# Comparison - Organ direction estimation

In [ ]:
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import open3d as o3d  # just to hide some unwanted messages

from IPython.display import clear_output
from scipy.spatial.distance import euclidean

from plantdb.commons.test_database import test_database
from plantdb.commons.io import read_graph
from plantdb.commons.io import read_point_cloud

from plant3dvision.visu.plotly import plotly_pointcloud
from plant3dvision.visu.plotly import plotly_mesh
from plant3dvision.visu.plotly import plotly_skeleton
from plant3dvision.visu.plotly import plotly_treegraph

## Connect to the database & get the data

### Connect to the database

In [ ]:
db = test_database('real_plant_analyzed', no_auth=True)
db.connect()

### Get the dataset

We now select the `real_plant` dataset for the demo:

In [ ]:
scan = db.get_scan("real_plant_analyzed")

### Get the PointCloud fileset

We now select the `PointCloud_1_0_1_0_10_0_7ee836e5a9` fileset for the demo:

In [ ]:
pcd_fs = scan.get_fileset("PointCloud_1_0_1_0_10_0_7ee836e5a9")
pcd_file = pcd_fs.get_file('PointCloud')

In [ ]:
pcd = read_point_cloud(pcd_file)

### Get the TreeGraph fileset

We now select the `TreeGraph__False_CurveSkeleton_c304a2cc71` fileset for the demo:

In [ ]:
tree_fs = scan.get_fileset("TreeGraph__False_CurveSkeleton_c304a2cc71")
tree_file = tree_fs.get_file('TreeGraph')

In [ ]:
tree = read_graph(tree_file)

We may now disconnect from the database as we will not need it anymore:

In [ ]:
db.disconnect()

## View the original data

We use Plotly library to visualize the loaded triangular mesh.

Have a look at *Plotly Open Source Graphing Library for Python* here: https://plotly.com/python/

In [ ]:
fig = plotly_treegraph(tree, height=800)
fig.show()

## Functions

In [ ]:
from plant3dvision.arabidopsis import get_fruit, get_nodes_by_label, fit_plane

In [ ]:
def stem_and_fruit_directions(tree, n_nodes_fruit=5, n_nodes_stem=5):
    """Tim/Julie method extracted from `plant3dvision.arabidopsis.compute_angles_and_internodes`."""
    unordered_main_stem = get_nodes_by_label(tree, "stem")
    unordered_branching_points = get_nodes_by_label(tree, "node")
    all_fruit_points = []
    node_info_list = []

    # re order nodes
    nodes_dict = {}
    for ubp in unordered_branching_points:
        nodes_dict[ubp] = tree.nodes[ubp]["fruit_id"]
    branching_points = [k for k, v in sorted(nodes_dict.items(), key=lambda item: item[1])]

    # re order main stem
    stem_dict = {}
    for umn in unordered_main_stem:
        stem_dict[umn] = tree.nodes[umn]["main_stem_id"]
    main_stem = [k for k, v in sorted(stem_dict.items(), key=lambda item: item[1])]

    for i in range(len(branching_points) - 1):
        node_point = np.array(tree.nodes[branching_points[i]]["position"])
        node_next_point = np.array(tree.nodes[branching_points[i + 1]]["position"])
        node_fruit_points = [np.array(tree.nodes[n]["position"]) for n in get_fruit(tree, i)]

        if len(node_fruit_points) > 1:
            vertices_fruit_plane_est = node_fruit_points[0:n_nodes_fruit]
            idx = main_stem.index(branching_points[i])
            stem_neighbors_id = main_stem[idx - n_nodes_stem // 2:idx + n_nodes_stem // 2]
            vertices_node_plane_est = [tree.nodes[stem_id]["position"] for stem_id in stem_neighbors_id]

            points = np.vstack([vertices_fruit_plane_est, vertices_node_plane_est])
            _, v1, v2 = fit_plane(points)

            fruit_points = np.vstack(node_fruit_points)
            fruit_mean = fruit_points.mean(axis=0)
            all_fruit_points.append(fruit_points.tolist())

            new_v1 = fruit_mean - node_point
            new_v1 = new_v1.dot(v1) * v1 + new_v1.dot(v2) * v2
            new_v1 /= np.linalg.norm(new_v1)

            # Set v1 as the fruit direction and v2 as the stem direction
            v1, v2 = new_v1, v2 - v2.dot(new_v1) * new_v1
            if v2.dot(node_next_point - node_point) < 0:
                v2 = - v2

            node_info_list.append({
                "node_point": node_point,
                "fruit_direction": v1,
                "stem_direction": v2
            })
    return node_info_list

In [ ]:
from plant3dvision.utils import flatten
from plant3dvision.proc3d import crop_point_cloud
from plant3dvision.tree import select_stem_nodes_by_euclidean_distance, nodes_coordinates
from plant3dvision.tree import select_fruit_nodes, get_ordered_branching_point_nodes
from plant3dvision.arabidopsis import compute_stem_and_fruit_directions


def plot_subtree_pcd_dir(tree, bp_node_id, pcd):
    branching_points = get_ordered_branching_point_nodes(tree)
    # Get the index of the selected node in the ordered branching points list
    bp_idx = branching_points.index(bp_node_id) -1

    # -- Get the subtree around branching point node:
    stem_nodes = select_stem_nodes_by_euclidean_distance(tree, bp_node_id, max_node_dist=15)
    fruit_nodes = select_fruit_nodes(tree, bp_node_id, max_node_dist=None)
    subtree = tree.subgraph(stem_nodes + list(flatten(fruit_nodes)))

    # -- Get the cropped pcd around branching point node:
    stem_points = nodes_coordinates(tree, stem_nodes)
    # Compute the bounding box to crop point cloud in Z:
    bbox = {"x": [], "y": [], "z": []}
    bbox["z"] = [np.min(stem_points, axis=0)[2], np.max(stem_points, axis=0)[2]]
    bbox["x"] = [np.min(pcd.points, axis=0)[0], np.max(pcd.points, axis=0)[0]]
    bbox["y"] = [np.min(pcd.points, axis=0)[1], np.max(pcd.points, axis=0)[1]]
    # Crop the point cloud:
    subpcd = crop_point_cloud(pcd, bbox)

    # - Plot sutree & PCD:
    subtree_data = plotly_treegraph(subtree, height=600, width=800,
                                    title=f"Tree graph around node {bp_node_id}.").data
    x, y, z = np.array(subpcd.points).T
    subpcd_data = go.Scatter3d(x=x, y=y, z=z, mode="markers", name="Point cloud",
                               marker=dict(size=1, color='green', opacity=0.8))
    fig = go.Figure(data=list(subtree_data) + [subpcd_data])
    fig.update_layout(height=600, width=800, title=f"Tree graph & point cloud around node {bp_node_id}.")
    fig.update_scenes(aspectmode='data')

    # BLUE = v1
    node_info =  stem_and_fruit_directions(tree)
    fruit_dir, stem_dir, bp_coord = node_info[bp_idx]["fruit_direction"], node_info[bp_idx]["stem_direction"], node_info[bp_idx]["node_point"]

    # Add estimated stem line as blue line:
    mini = euclidean(bp_coord, stem_points[0])  # distance to the lowest stem point
    maxi = euclidean(bp_coord, stem_points[-1])  # distance to the highest stem point
    linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + bp_coord
    x, y, z = linepts.T
    fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction",
                      marker=dict(color='blue', size=3, opacity=1, symbol="diamond"))
    # Add estimated fruit line as blue line:
    linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + bp_coord
    x, y, z = linepts.T
    fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Fruit direction",
                      marker=dict(color='blue', size=3, opacity=1, symbol="diamond"), line=dict(width=5))

    # RED = v2
    fruit_dirs, stem_dirs, bp_coords, fruit_pts = compute_stem_and_fruit_directions(subtree)
    stem_dir, bp_coord = stem_dirs[0], bp_coords[0]
    # Add estimated stem line as red line:
    mini = euclidean(bp_coord, stem_points[0])  # distance to the lowest stem point
    maxi = euclidean(bp_coord, stem_points[-1])  # distance to the highest stem point
    linepts = stem_dir * np.mgrid[-mini:maxi:2j][:, np.newaxis] + bp_coord
    x, y, z = linepts.T
    fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name="Stem direction (Jo)",
                      marker=dict(color='red', size=3, opacity=1, symbol="diamond"))
    for n, fruit_dir in enumerate(fruit_dirs):
        # Add estimated fruit line as red line:
        linepts = fruit_dir * np.mgrid[0:20:2j][:, np.newaxis] + bp_coord
        x, y, z = linepts.T
        fig.add_scatter3d(x=x, y=y, z=z, mode="markers+lines", name=f"Fruit {n} direction (Jo)",
                          marker=dict(color='red', size=3, opacity=1, symbol="diamond"), line=dict(width=5))

    return fig

## Visualize differences in estimation

Graphs are interactive.

You may click on some items of the legend to turn them off/on.

Double clicks will display only the items.

In [ ]:
plot_subtree_pcd_dir(tree, 869, pcd)

In [ ]:
plot_subtree_pcd_dir(tree, 585, pcd)